### 📘 Deep Learning RAG Project

**Assignment Topic:** Design and implement a simple Retrieval-Augmented Generation (RAG) system that answers user questions using custom documents or the Hugging Face OpenRAGBench dataset. The system should retrieve the most relevant document passages using semantic embeddings and vector similarity search, then generate context-aware answers using a pre-trained language model.

### 🧠 Problem Statement

`Develop a simple Retrieval-Augmented Generation (RAG) system to answer questions from custom documents. Build a pipeline that retrieves relevant information from a document and uses a language model to generate answers.`

---

Import Libraries

In [1]:
import numpy as np
import torch
import os
import json

from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss

Load HuggingFace Dataset

In [2]:
import os
import json

BASE_PATH = "open_ragbench/pdf/arxiv"

with open(os.path.join(BASE_PATH, "queries.json"), "r", encoding="utf-8") as f:
    queries = json.load(f)

with open(os.path.join(BASE_PATH, "answers.json"), "r", encoding="utf-8") as f:
    answers = json.load(f)

with open(os.path.join(BASE_PATH, "qrels.json"), "r", encoding="utf-8") as f:
    qrels = json.load(f)

print("Queries:", len(queries))
print("Answers:", len(answers))
print("Qrels:", len(qrels))

Queries: 3045
Answers: 3045
Qrels: 3045


Load Corpus

In [3]:
corpus_path = os.path.join(BASE_PATH, "corpus")

documents = []

for file in os.listdir(corpus_path):
    if file.endswith(".json"):
        with open(os.path.join(corpus_path, file), "r", encoding="utf-8") as f:
            doc = json.load(f)
            documents.append(doc)

print("Documents loaded:", len(documents))

Documents loaded: 1000


Explore Dataset

In [4]:
print(next(iter(queries.values())))

{'query': 'What are the challenges in estimating output impedance in inverter-based grids?', 'type': 'abstractive', 'source': 'text-image'}


Extract Documents

In [5]:
import os
import json

corpus_path = "open_ragbench/pdf/arxiv/corpus"

text_documents = []

for filename in os.listdir(corpus_path):
    if filename.endswith(".json"):
        with open(os.path.join(corpus_path, filename), "r", encoding="utf-8") as f:
            doc = json.load(f)
        text = ""
        if doc.get("title"):
            text += doc["title"] + "\n\n"
        if doc.get("abstract"):
            text += doc["abstract"] + "\n\n"
        if "sections" in doc:
            for section in doc["sections"]:
                if isinstance(section, dict):
                    if section.get("heading"):
                        text += section["heading"] + "\n"

                    if section.get("text"):
                        text += section["text"] + "\n\n"

        text_documents.append(text)

print("Number of documents:", len(text_documents))

Number of documents: 1000


Test Run For Embedding

In [6]:
print(documents[0].keys())

dict_keys(['title', 'sections', 'id', 'authors', 'categories', 'abstract', 'updated', 'published'])


Load Embedding Model

In [7]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Create Embeddings

In [8]:
document_embeddings = embedding_model.encode(
    text_documents,
    convert_to_numpy=True,
    show_progress_bar=True
)
print(document_embeddings.shape)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

(1000, 384)


Build FAISS Index

In [9]:
dimension = document_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(document_embeddings)
print("Documents Indexed :", index.ntotal)

Documents Indexed : 1000


Load Generator

In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

User Question

In [11]:
query = "What is Retrieval Augmented Generation?"

Convert Query into Embedding

In [12]:
query_embedding = embedding_model.encode([query], convert_to_numpy=True)

Retrieve Top 3 Documents

In [13]:
k = 3
distances, indices = index.search(query_embedding, k)
retrieved_docs = []
for idx in indices[0]:
    retrieved_docs.append(text_documents[idx])

Display Retrieved Documents

In [14]:
for i, doc in enumerate(retrieved_docs):
    print("Document", i + 1)
    print(doc)
    print("-" * 60)

Document 1
Enhancing Scientific Reproducibility Through Automated BioCompute Object
  Creation Using Retrieval-Augmented Generation from Publications

The exponential growth in computational power and accessibility has
transformed the complexity and scale of bioinformatics research, necessitating
standardized documentation for transparency, reproducibility, and regulatory
compliance. The IEEE BioCompute Object (BCO) standard addresses this need but
faces adoption challenges due to the overhead of creating compliant
documentation, especially for legacy research. This paper presents a novel
approach to automate the creation of BCOs from scientific papers using
Retrieval-Augmented Generation (RAG) and Large Language Models (LLMs). We
describe the development of the BCO assistant tool that leverages RAG to
extract relevant information from source papers and associated code
repositories, addressing key challenges such as LLM hallucination and
long-context understanding. The implementation i

Create Prompt

In [15]:
MAX_DOCS = 3

context = "\n\n".join(doc[:1000] for doc in retrieved_docs[:MAX_DOCS])

prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{query}

Answer:
"""

Generate Answer

In [16]:
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

outputs = model.generate(
    **inputs,
    max_new_tokens=150
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)

introduce a novel approach to automate the creation of BCOs from scientific papers using Retrieval-Augmented Generation (RAG) and Large Language Models (LLMs).


Test Multiple Questions

In [17]:
questions = [
    "Explain Retrieval Augmented Generation.",
    "How does retrieval improve generation?",
    "What is the purpose of embeddings?",
]

for question in questions:

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )

    distances, indices = index.search(query_embedding, 3)

    context = "\n\n".join(
        text_documents[i][:500]
        for i in indices[0]
    )

    prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=120
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("Question:", question)
    print("Answer:", answer)
    print("=" * 80)

Question: Explain Retrieval Augmented Generation.
Answer: Automated BioCompute Object Creation Using Retrieval-Augmented Generation from Publications
Question: How does retrieval improve generation?
Answer: Automated BioCompute Object Creation Using Retrieval-Augmented Generation from Publications
Question: What is the purpose of embeddings?
Answer: to enable massive density and durability


Conclusion

- Loaded OpenRAGBench dataset.
- Generated embeddings using SentenceTransformer.
- Stored embeddings in FAISS.
- Retrieved relevant documents using similarity search.
- Generated answers using FLAN-T5.
- Successfully implemented a simple Retrieval-Augmented Generation (RAG) pipeline.

---
### 🏁 Submission Checklist

- ✅ Markdown are used with proper description.
- ✅ All plots render with labels and titles.
- ✅ Notebook runs cleanly from top to bottom (`Kernel → Restart & Run All`)

**Saved as:** `week7_Amulya_Shrivastava.ipynb`